# RQ3-02 — Retrain Traditional ML on Leakage-Free Data

Notebook này bám phương pháp nền của `05_LG_RF_LGBM_XGB_SVM.ipynb`:

- Input: `data_with_features_minus_1000_human.csv`.
- Split **80% Train / 20% internal Test**, stratified, `SEED=42`.
- TF-IDF: `max_features=5000`, `ngram_range=(1,2)`, fit trên Train.
- 8 handcrafted features, StandardScaler fit trên Train.
- 5-fold Stratified CV trên Train.
- Models/params nền: Logistic Regression, Random Forest, LightGBM, XGBoost, SVM-RBF.
- Train model chính thức trên 80% Train và đánh giá internal 20% Test.
- Save vectorizer/scaler/models để notebook RQ3 cuối chỉ **inference** trên external benchmark.

> External 1,000 Human + 1,000 AI **không được đọc trong notebook này**.

In [ ]:
# Nếu môi trường thiếu package:
# %pip install -U pandas numpy scipy scikit-learn lightgbm xgboost joblib matplotlib seaborn

import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

SEED = 42
TEST_SIZE = 0.20
RUN_SVM = True  # SVM RBF đúng notebook nền nhưng có thể chạy lâu.

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd
if Path("/content/AI-Generated-Text-Essay-Detection").exists():
    PROJECT_ROOT = Path("/content/AI-Generated-Text-Essay-Detection")

DATA_CANDIDATES = [
    PROJECT_ROOT / "data" / "processed" / "data_with_features_minus_1000_human.csv",
    PROJECT_ROOT / "data_with_features_minus_1000_human.csv",
    cwd / "data_with_features_minus_1000_human.csv",
]
DATA_FILE = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_FILE is None:
    raise FileNotFoundError("Không tìm thấy data_with_features_minus_1000_human.csv")

RESULT_DIR = PROJECT_ROOT / "results" / "rq3_retrain_ml"
MODEL_DIR = RESULT_DIR / "models"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_FILE :", DATA_FILE)
print("RESULT_DIR:", RESULT_DIR)

In [ ]:
df = pd.read_csv(DATA_FILE).copy()
required = {"label", "text", "avg_word_length", "punctuation_ratio", "burstiness", "word_count", "avg_sentence_length", "lexical_diversity", "sentence_count", "function_word_ratio"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Dataset thiếu cột: {missing}")

df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)
df["_row_id"] = np.arange(len(df))

feature_cols = [
    "avg_word_length", "punctuation_ratio", "burstiness",
    "word_count", "avg_sentence_length", "lexical_diversity",
    "sentence_count", "function_word_ratio",
]

y = df["label"]
train_idx, test_idx = train_test_split(
    df.index,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=SEED,
)

train_df = df.loc[train_idx].reset_index(drop=True)
test_df = df.loc[test_idx].reset_index(drop=True)

print("Total:", len(df))
print("Train:", len(train_df), train_df["label"].value_counts().to_dict())
print("Test :", len(test_df), test_df["label"].value_counts().to_dict())
assert set(train_df["_row_id"]).isdisjoint(set(test_df["_row_id"]))

manifest = pd.concat([
    train_df[["_row_id", "label"]].assign(split="train_80"),
    test_df[["_row_id", "label"]].assign(split="test_20"),
], ignore_index=True)
manifest.to_csv(RESULT_DIR / "internal_split_manifest.csv", index=False)

In [ ]:
# Feature processing giống notebook 05: fit chỉ trên Train.
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(train_df["text"])
X_test_tfidf = vectorizer.transform(test_df["text"])

scaler = StandardScaler()
X_train_feat_scaled = scaler.fit_transform(train_df[feature_cols])
X_test_feat_scaled = scaler.transform(test_df[feature_cols])

X_train = hstack([X_train_tfidf, csr_matrix(X_train_feat_scaled)]).tocsr()
X_test = hstack([X_test_tfidf, csr_matrix(X_test_feat_scaled)]).tocsr()
y_train = train_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

joblib.dump(vectorizer, MODEL_DIR / "tfidf_vectorizer.joblib")
joblib.dump(scaler, MODEL_DIR / "feature_scaler.joblib")
(RESULT_DIR / "feature_cols.json").write_text(json.dumps(feature_cols, indent=2), encoding="utf-8")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("scale_pos_weight:", scale_pos_weight)

In [ ]:
def evaluate_and_save(model_name, model, cv_jobs=1, score_mode="proba"):
    print("\n" + "=" * 80)
    print(model_name)
    print("=" * 80)
    start = time.time()

    cv_scores = cross_val_score(
        model, X_train, y_train,
        cv=skf, scoring="f1", n_jobs=cv_jobs,
    )
    print(f"CV F1: {cv_scores.mean():.6f} ± {cv_scores.std():.6f}")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    if score_mode == "proba":
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_score),
        "CV_F1_mean": cv_scores.mean(),
        "CV_F1_std": cv_scores.std(),
        "Seconds": time.time() - start,
    }

    safe = model_name.lower().replace(" ", "_").replace("-", "_")
    joblib.dump(model, MODEL_DIR / f"{safe}.joblib")
    pd.DataFrame({
        "_row_id": test_df["_row_id"],
        "label": y_test,
        "prediction": y_pred,
        "score_ai": y_score,
        "text": test_df["text"],
    }).to_csv(RESULT_DIR / f"{safe}_internal_test_predictions.csv", index=False)

    print(metrics)
    return metrics

In [ ]:
all_results = []

all_results.append(evaluate_and_save(
    "Logistic Regression",
    LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED),
    cv_jobs=1,
))

all_results.append(evaluate_and_save(
    "Random Forest",
    RandomForestClassifier(
        n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1
    ),
    cv_jobs=1,
))

all_results.append(evaluate_and_save(
    "LightGBM",
    LGBMClassifier(
        n_estimators=200, class_weight="balanced", random_state=SEED, verbose=-1
    ),
    cv_jobs=1,
))

all_results.append(evaluate_and_save(
    "XGBoost",
    XGBClassifier(
        n_estimators=200,
        scale_pos_weight=scale_pos_weight,
        random_state=SEED,
        eval_metric="logloss",
    ),
    cv_jobs=1,
))

if RUN_SVM:
    all_results.append(evaluate_and_save(
        "SVM",
        SVC(
            kernel="rbf",
            class_weight="balanced",
            random_state=SEED,
            cache_size=2000,
            shrinking=True,
        ),
        cv_jobs=-1,
        score_mode="decision",
    ))
else:
    print("SVM skipped: RUN_SVM=False")

In [ ]:
results_df = pd.DataFrame(all_results).sort_values("F1", ascending=False)
summary_path = RESULT_DIR / "ml_internal_results.csv"
results_df.to_csv(summary_path, index=False)

display(results_df)
print("Saved:", summary_path)
print("Saved models:", MODEL_DIR)
print("\n✅ Retrain ML hoàn tất. Không có RQ3 external data nào được dùng trong training.")